<a href="https://colab.research.google.com/github/JoyeeChen/DeepLearningAndAIAlignmentProjects/blob/main/AHA2ScoreAfterFinetuningAndRLAIF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is for https://www.compassionml.com/results-and-news#h.y8rxhqvrp2qp

In [1]:
!pip install wandb -qqq
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 38 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (788 kB/s)
Selecting previously unselected package tree.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
import os
import wandb

In [3]:
#important: For auto-logging of models as artifacts: just set environment varilable WANDB_LOG_MODEL to true!
#advice taken from https://colab.research.google.com/github/wandb/examples/blob/master/colabs/huggingface/Optimize_Hugging_Face_models_with_Weights_&_Biases.ipynb#scrollTo=dJlW9xrS6VFj
%env WANDB_LOG_MODEL=true

env: WANDB_LOG_MODEL=true


In [4]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chen-joyee (chen-joyee-compassion-in-machine-learning) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
from huggingface_hub import login
from google.colab import userdata
HF_TOKEN=userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")


Successfully logged in to Hugging Face!


In [6]:
!pip install unsloth
from unsloth import FastLanguageModel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.9/346.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 157.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.2/269.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [10]:
with wandb.init(
    project = "14092025",
    notes = "Code related to https://www.compassionml.com/results-and-news#h.y8rxhqvrp2qp",
) as run:
    print("Project initialized!")

    #important: For auto-logging of models as artifacts: just set environment varilable WANDB_LOG_MODEL to true!
    #advice taken from https://colab.research.google.com/github/wandb/examples/blob/master/colabs/huggingface/Optimize_Hugging_Face_models_with_Weights_&_Biases.ipynb#scrollTo=dJlW9xrS6VFj

    #Or just let the base_model artifact contain only the name without the actual bits and bytes.

    # Load model directly
    base_model_name = "meta-llama/Llama-3.1-8B"
    base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

    #max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
    #dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
    #load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.
    #base_model_name = "meta-llama/Llama-3.1-8B"

    #base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    #model_name = base_model_name,
    #max_seq_length = max_seq_length,
    #dtype = dtype,
    #load_in_4bit = load_in_4bit,
    #)
    #run.log_model(
    #
    #)
    base_model.save_pretrained(save_directory="./" + base_model_name)







Project initialized!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
with wandb.init(
    project = "14092025",
    notes = "Code related to https://www.compassionml.com/results-and-news#h.y8rxhqvrp2qp",
) as run:
    #base_model_artifact = wandb.Artifact(
    #    name = "base_model",
    #    type = "model",
    #)
    run.log_model(
        path = "./" + base_model_name,
        name = "base_model"
    )

wandb: Adding directory to artifact (meta-llama/Llama-3.1-8B)... Done. 108.0s


In [ ]:
#this codeblock is focused on creating the config settings for FPTs.
with wandb.init(
    project = "14092025",
    notes = "Code related to https://www.compassionml.com/results-and-news#h.y8rxhqvrp2qp",
) as run:
    #Joyee's research:
    #Basellama_plus_3kv3 is here https://huggingface.co/CompassioninMachineLearning/Basellama_plus3kv3
    #And it has config files https://huggingface.co/CompassioninMachineLearning/Basellama_plus3kv3/blob/main/config.json
    #And generation config files https://huggingface.co/CompassioninMachineLearning/Basellama_plus3kv3/blob/main/generation_config.json
    #Meanwhile, for the code previously used, we couldn't get a perfect match,
    #but I suspect the closest would be the internal colab Llama_(8B)-pretraining.ipynb

    #Now that internal colab itself has different entry points for parameters:

    #First entry point:
    #model, tokenizer = FastLanguageModel.from_pretrained(
        #model_name =
        #max_seq_length =
        #dtype =
        #load_in_4bit =
    #)

    #Second entry point:
    #model = FastLanguageModel.get_peft_model(
    #model,
    #r =
    #target_modules =
    #lora_alpha =
    #lora_dropout =
    #bias =
    #use_gradient_checkpointing =
    #random_state =
    #use_rslora =
    #loftq_config =
    #)

    #Third entry point:
    #dataset = load_dataset(

    #Fourth entry point:
    #trainer = UnslothTrainer(
    #model = model,
    #tokenizer = tokenizer,
    #train_dataset = dataset,
    #dataset_text_field = "text",
    #max_seq_length =
    #dataset_num_proc =

    #args = UnslothTrainingArguments(
    #    per_device_train_batch_size =
    #    gradient_accumulation_steps =

    #    warmup_ratio =
    #    num_train_epochs =

    #    learning_rate =
    #    embedding_learning_rate =

    #    logging_steps =
    #    optim =
    #    weight_decay =
    #    lr_scheduler_type =
    #    seed =
    #    output_dir =
    #    report_to =
    #),
    #)

    master_config = {
        model_name = ,
        max_sequence_length = ,

    }


    config = {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.55.4",
  "unsloth_fixed": true,
  "unsloth_version": "2025.9.4",
  "use_cache": true,
  "vocab_size": 128256
}
    fpt_config = wandb.Artifact(

    )
    generation_config = {
  "_from_model_config": true,
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "max_length": 131072,
  "pad_token_id": 128004,
  "temperature": 0.6,
  "top_p": 0.9,
  "transformers_version": "4.55.4"
}
    fpt_generation_config = wandb.Artifact(

    )

In [10]:
#this codeblock is if, provisionally, we should just load in the 3k-FPTed model we already have without worrying about how we created it.
with wandb.init(
    project = "14092025",
    notes = "Code related to https://www.compassionml.com/results-and-news#h.y8rxhqvrp2qp",
) as run:
    #Joyee's research:
    Basellama_plus_3kv3_tokenizer = AutoTokenizer.from_pretrained("CompassioninMachineLearning/Basellama_plus3kv3")
    Basellama_plus_3kv3_model = AutoModelForCausalLM.from_pretrained("CompassioninMachineLearning/Basellama_plus3kv3")
    Basellama_plus_3kv3_model.save_pretrained(save_directory="./" + "CompassioninMachineLearning/Basellama_plus3kv3")
    run.log_model(
        path = "./" + "CompassioninMachineLearning/Basellama_plus3kv3",
        name = "Basellama_plus3kv3"
    )

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

wandb: Adding directory to artifact (CompassioninMachineLearning/Basellama_plus3kv3)... Done. 94.1s


In [ ]:
def fpt():

In [ ]:
def fpt_and_log():

In [ ]:
def sft():

In [ ]:
def sft_and_log():

In [ ]:
def rlaif():

In [ ]:
def rlaif_and_log():

In [ ]:
def evaluate():

In [ ]:
def evaluate_and_log():